# HYPERVIEW2 Sensor-Aware Output Calibration

Minimalny eksperyment koncowy: sprawdzamy, czy prosta kalibracja wyjscia dekodera usuwa czesc domain shiftu HySpecNet -> HYPERVIEW2.

Pipeline:

1. generujemy albo ladujemy rekonstrukcje Mamby HySpecNet-202 na HYPERVIEW2 (`230 -> 202 -> Mamba -> 230`),
2. generujemy baseline `spectral_resample_passthrough` (`230 -> 202 -> 230`, bez modelu),
3. fitujemy per-band affine calibration tylko na train split downstream,
4. porownujemy downstream `original_train_to_recon_val`, `recon_train_to_recon_val`, per-band bias i prediction shift.

To jest diagnostyka transferu HYPERVIEW2/PRISMA, nie reference-comparable wynik HySpecNet-11k.

## 1. Ustawienia

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/mhx1467/master-thesis-code.git'
REPO_DIR = Path('/content/hsi')
REPO_REF = 'main'

DRIVE_HSI = Path('/content/drive/MyDrive/hsi')
DRIVE_HV2_ROOT = DRIVE_HSI / 'data/hyperview2/HYPERVIEW2'
DRIVE_CHECKPOINTS = DRIVE_HSI / 'checkpoints'
DRIVE_RECONS = DRIVE_HSI / 'reconstructions/hyperview2_sensor_calibration'
DRIVE_RESULTS = DRIVE_HSI / 'downstream_results/hyperview2_sensor_calibration'

SOURCE_CKPT_CANDIDATES = [
    DRIVE_CHECKPOINTS / 'hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_001_spectral_feature_ft_best.pt',
    DRIVE_CHECKPOINTS / 'hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_001_ft_best.pt',
    DRIVE_CHECKPOINTS / 'hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_01_best.pt',
]

EVAL_PRESET = 'quick'  # quick | full
PRESET = {
    'quick': {
        'regressors': ['extra_trees'],
        'downstream_train_samples': 600,
        'downstream_val_samples': 150,
        'calibration_samples': 600,
        'per_band_samples': 120,
    },
    'full': {
        'regressors': ['hist_gradient_boosting', 'extra_trees', 'random_forest'],
        'downstream_train_samples': None,
        'downstream_val_samples': None,
        'calibration_samples': None,
        'per_band_samples': None,
    },
}
if EVAL_PRESET not in PRESET:
    raise ValueError('EVAL_PRESET must be quick or full')

RUN_SOURCE_RECONSTRUCTION = True
RUN_RESAMPLE_CONTROL = True
RUN_CALIBRATION = True
RUN_DOWNSTREAM = True
RECOMPUTE_SOURCE_RECONSTRUCTION = False
RECOMPUTE_RESAMPLE_CONTROL = False
RECOMPUTE_CALIBRATION = True
RECOMPUTE_REGRESSORS = True
REQUIRE_CUDA = True
FORCE_REINSTALL_ENV = False

SEED = 42
VAL_FRACTION = 0.2
MODALITY = 'prisma'
NORMALIZATION = 'reflectance_0_1'
FEATURE_SET = 'mean_std_derivatives'
SPECTRAL_MAPPING = 'hyspecnet_202_approx'
COLAB_DATALOADER_NUM_WORKERS = 0
RECON_BATCH_SIZE = 1
CALIBRATION_BATCH_SIZE = 32
RESAMPLE_BATCH_SIZE = 32
FEATURE_BATCH_SIZE = 128

SOURCE_VARIANT = 'mamba_hyspecnet202_uncalibrated'
RESAMPLE_VARIANT = 'spectral_resample_passthrough_hyspecnet202_to_230'
CALIBRATED_VARIANT = 'mamba_hyspecnet202_sensor_affine_calibrated'

print('Preset:', EVAL_PRESET)
print(PRESET[EVAL_PRESET])


## 2. Repo

In [ ]:
import urllib.request

BOOTSTRAP_URL = (
    'https://raw.githubusercontent.com/mhx1467/master-thesis-code/'
    f'{REPO_REF}/scripts/colab_bootstrap.py'
)
exec(urllib.request.urlopen(BOOTSTRAP_URL).read().decode('utf-8'))
sync_repo(REPO_URL, REPO_DIR, REPO_REF)


## 3. Zaleznosci

Komorka instaluje Mambę z prebuilt wheels zgodnych z Colab Python 3.12 / Torch 2.7.1 CUDA 12.6. Po pierwszej instalacji runtime zostanie zrestartowany; potem uruchom notebook ponownie od komorki repo.

In [ ]:
install_mamba_env(
    marker_name='.hsi_compression_hv2_calibration_env_v2_torch27_mamba232',
    force_reinstall=FORCE_REINSTALL_ENV,
    require_cuda=REQUIRE_CUDA,
    error_message='mamba-ssm is required when RUN_SOURCE_RECONSTRUCTION=True.',
)


## 4. Drive, dane, checkpoint

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
for directory in [DRIVE_CHECKPOINTS, DRIVE_RECONS, DRIVE_RESULTS]:
    directory.mkdir(parents=True, exist_ok=True)


def is_hyperview2_root(path: Path) -> bool:
    required = [
        path / 'train_gt.csv',
        path / 'submission.csv',
        path / 'train/hsi_satellite',
        path / 'test/hsi_satellite',
    ]
    return all(item.exists() for item in required)


def find_hyperview2_root(search_root: Path) -> Path | None:
    if is_hyperview2_root(search_root):
        return search_root
    if search_root.exists():
        for candidate in sorted(search_root.rglob('HYPERVIEW2')):
            if is_hyperview2_root(candidate):
                return candidate
    return None

HV2_ROOT = find_hyperview2_root(DRIVE_HV2_ROOT.parent)
if HV2_ROOT is None:
    raise FileNotFoundError(f'Nie znaleziono HYPERVIEW2 pod {DRIVE_HV2_ROOT.parent}')

SOURCE_CKPT = next((path for path in SOURCE_CKPT_CANDIDATES if path.exists()), None)
if SOURCE_CKPT is None:
    print('Checked candidates:')
    for path in SOURCE_CKPT_CANDIDATES:
        print(' -', path)
    raise FileNotFoundError('Brakuje checkpointu HySpecNet Mamba na Drive.')

print('HV2_ROOT:', HV2_ROOT)
print('SOURCE_CKPT:', SOURCE_CKPT)
for rel in ['train/hsi_satellite', 'test/hsi_satellite']:
    directory = HV2_ROOT / rel
    print(f'{rel:24s} {len(list(directory.glob("*.npz"))):5d} npz files')


## 5. Importy i split downstream

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from hsi_compression.downstream import build_hyperview2_samples, split_samples
from hsi_compression.downstream.hyperview2_compression_eval import (
    CompressionCheckpoint,
    best_by_variant_mode,
    compute_per_band_diagnostics,
    evaluate_downstream_regressors,
    prediction_metrics_by_target,
    prediction_shift_decomposition,
    read_reconstruction_summary,
    reconstruct_affine_calibrated_reconstruction,
    reconstruct_checkpoint,
    reconstruct_spectral_resample_passthrough,
    save_downstream_artifacts,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
print('DEVICE:', DEVICE)

samples = build_hyperview2_samples(HV2_ROOT, modality=MODALITY, split='train')
train_samples, val_samples = split_samples(samples, val_fraction=VAL_FRACTION, seed=SEED)


def limit_samples_spread(items, max_samples):
    items = list(items)
    if max_samples is None or max_samples <= 0 or max_samples >= len(items):
        return items
    indices = np.linspace(0, len(items) - 1, num=int(max_samples), dtype=np.int64)
    return [items[int(index)] for index in indices]

calibration_samples = limit_samples_spread(train_samples, PRESET[EVAL_PRESET]['calibration_samples'])
calibration_ids = [sample.sample_id for sample in calibration_samples]
print('Full train/val:', len(train_samples), len(val_samples))
print('Calibration train IDs:', len(calibration_ids))
print('First calibration IDs:', calibration_ids[:10])


## 6. Rekonstrukcja Mamby i resampling-only

In [ ]:
recon_roots = {}
recon_summaries = {}

source_root = DRIVE_RECONS / SOURCE_VARIANT / 'HYPERVIEW2'
if source_root.exists() and not RECOMPUTE_SOURCE_RECONSTRUCTION:
    print('Using existing source reconstruction:', source_root)
    source_summary = read_reconstruction_summary(source_root) or {}
else:
    if not RUN_SOURCE_RECONSTRUCTION:
        raise FileNotFoundError(f'Missing source reconstruction: {source_root}')
    checkpoint = CompressionCheckpoint(
        name='mamba_hyspecnet202_source',
        path=SOURCE_CKPT,
        variant_name=SOURCE_VARIANT,
        modality=MODALITY,
        compression_normalization=NORMALIZATION,
        recon_feature_normalization='none',
        batch_size=RECON_BATCH_SIZE,
        num_workers=COLAB_DATALOADER_NUM_WORKERS,
        use_bitstream=True,
        pad_multiple=4,
        min_spatial_size=4,
        allow_in_channel_adapter=False,
        spectral_mapping=SPECTRAL_MAPPING,
    )
    source_root, source_summary = reconstruct_checkpoint(
        checkpoint,
        source_root=HV2_ROOT,
        recon_parent=DRIVE_RECONS,
        split='train',
        device=DEVICE,
        checkpoint_normalization_fallback=NORMALIZATION,
    )
recon_roots[SOURCE_VARIANT] = source_root
recon_summaries[SOURCE_VARIANT] = source_summary

resample_root = DRIVE_RECONS / RESAMPLE_VARIANT / 'HYPERVIEW2'
if resample_root.exists() and not RECOMPUTE_RESAMPLE_CONTROL:
    print('Using existing resampling control:', resample_root)
    resample_summary = read_reconstruction_summary(resample_root) or {}
elif RUN_RESAMPLE_CONTROL:
    resample_root, resample_summary = reconstruct_spectral_resample_passthrough(
        source_root=HV2_ROOT,
        recon_parent=DRIVE_RECONS,
        device=DEVICE,
        variant_name=RESAMPLE_VARIANT,
        modality=MODALITY,
        normalization=NORMALIZATION,
        spectral_mapping_name=SPECTRAL_MAPPING,
        batch_size=RESAMPLE_BATCH_SIZE,
        num_workers=COLAB_DATALOADER_NUM_WORKERS,
        split='train',
    )
else:
    raise FileNotFoundError(f'Missing resampling control: {resample_root}')
recon_roots[RESAMPLE_VARIANT] = resample_root
recon_summaries[RESAMPLE_VARIANT] = resample_summary

summary_rows = []
for variant, summary in recon_summaries.items():
    summary_rows.append({
        'variant': variant,
        'samples': summary.get('samples'),
        'masked_mse': summary.get('masked_mse'),
        'masked_mae': summary.get('masked_mae'),
        'masked_psnr': summary.get('masked_psnr', summary.get('masked_psnr_db')),
        'masked_sam_deg': summary.get('masked_sam_deg'),
        'actual_bpppc': summary.get('actual_bpppc'),
        'actual_bpppc_model_input': summary.get('actual_bpppc_model_input'),
    })
display(pd.DataFrame(summary_rows))


## 7. Sensor-aware affine calibration

In [ ]:
calibrated_root = DRIVE_RECONS / CALIBRATED_VARIANT / 'HYPERVIEW2'
if calibrated_root.exists() and not RECOMPUTE_CALIBRATION:
    print('Using existing calibrated reconstruction:', calibrated_root)
    calibrated_summary = read_reconstruction_summary(calibrated_root) or {}
elif RUN_CALIBRATION:
    calibrated_root, calibrated_summary = reconstruct_affine_calibrated_reconstruction(
        original_root=HV2_ROOT,
        recon_root=source_root,
        recon_parent=DRIVE_RECONS,
        variant_name=CALIBRATED_VARIANT,
        calibration_sample_ids=calibration_ids,
        device=DEVICE,
        source_variant=SOURCE_VARIANT,
        modality=MODALITY,
        original_normalization=NORMALIZATION,
        recon_normalization='none',
        batch_size=CALIBRATION_BATCH_SIZE,
        num_workers=COLAB_DATALOADER_NUM_WORKERS,
        split='train',
    )
else:
    raise FileNotFoundError(f'Missing calibrated reconstruction: {calibrated_root}')

recon_roots[CALIBRATED_VARIANT] = calibrated_root
recon_summaries[CALIBRATED_VARIANT] = calibrated_summary

calibration_view = {
    key: calibrated_summary.get(key)
    for key in [
        'calibration_sample_count', 'masked_mse', 'masked_mae', 'masked_psnr', 'masked_sam_deg',
        'scale_mean', 'scale_std', 'scale_min', 'scale_max',
        'offset_mean', 'offset_std', 'offset_min', 'offset_max',
    ]
}
display(pd.DataFrame([calibration_view]))


## 8. Downstream evaluation

In [ ]:
results_path = DRIVE_RESULTS / f'compression_downstream_summary_{EVAL_PRESET}.csv'
predictions_path = DRIVE_RESULTS / f'compression_downstream_predictions_{EVAL_PRESET}.csv'
metrics_path = DRIVE_RESULTS / f'compression_downstream_metrics_{EVAL_PRESET}.json'

if RUN_DOWNSTREAM and (RECOMPUTE_REGRESSORS or not results_path.exists()):
    results_df, predictions_df, metrics_payload = evaluate_downstream_regressors(
        hv2_root=HV2_ROOT,
        recon_roots=recon_roots,
        recon_feature_normalizations={name: 'none' for name in recon_roots},
        model_names=PRESET[EVAL_PRESET]['regressors'],
        modality=MODALITY,
        feature_set=FEATURE_SET,
        original_feature_normalization=NORMALIZATION,
        val_fraction=VAL_FRACTION,
        seed=SEED,
        n_jobs=-1,
        feature_device=DEVICE,
        feature_batch_size=FEATURE_BATCH_SIZE,
        feature_num_workers=COLAB_DATALOADER_NUM_WORKERS,
        max_train_samples=PRESET[EVAL_PRESET]['downstream_train_samples'],
        max_val_samples=PRESET[EVAL_PRESET]['downstream_val_samples'],
        verbose=True,
    )
    metrics_payload['reconstruction_summaries'] = recon_summaries
    metrics_payload['calibration'] = {
        'variant': CALIBRATED_VARIANT,
        'source_variant': SOURCE_VARIANT,
        'calibration_sample_ids': calibration_ids,
        'calibration_sample_count': len(calibration_ids),
        'eval_preset': EVAL_PRESET,
    }
    save_downstream_artifacts(DRIVE_RESULTS, results_df, predictions_df, metrics_payload)
    results_df.to_csv(results_path, index=False)
    predictions_df.to_csv(predictions_path, index=False)
    metrics_path.write_text(json.dumps(metrics_payload, indent=2, default=str), encoding='utf-8')
    print('Saved:', results_path)
else:
    print('Loading cached downstream results:', results_path)
    results_df = pd.read_csv(results_path)
    predictions_df = pd.read_csv(predictions_path) if predictions_path.exists() else pd.DataFrame()

best = best_by_variant_mode(results_df)
display(best[['variant', 'mode', 'model', 'hyperview_score', 'mean_mse', 'mean_mae']].sort_values(['mode', 'hyperview_score']))

main = best[best['mode'].eq('original_train_to_recon_val')].copy()
display(main[['variant', 'model', 'hyperview_score', 'mean_mse', 'mean_mae']].sort_values('hyperview_score'))


## 9. Czy kalibracja faktycznie pomogla?

In [ ]:
def score_for(variant, mode='original_train_to_recon_val'):
    rows = best[(best['variant'] == variant) & (best['mode'] == mode)]
    if rows.empty:
        return np.nan
    return float(rows.iloc[0]['hyperview_score'])

original_score = score_for('original', 'original_train_to_original_val')
resample_score = score_for(RESAMPLE_VARIANT)
source_score = score_for(SOURCE_VARIANT)
calibrated_score = score_for(CALIBRATED_VARIANT)

comparison = pd.DataFrame([
    {'metric': 'original', 'score': original_score},
    {'metric': 'resampling_only', 'score': resample_score},
    {'metric': 'source_mamba', 'score': source_score},
    {'metric': 'calibrated_mamba', 'score': calibrated_score},
    {'metric': 'calibrated_minus_source', 'score': calibrated_score - source_score},
    {'metric': 'calibrated_minus_resampling_only', 'score': calibrated_score - resample_score},
])
display(comparison)

fig, ax = plt.subplots(figsize=(8, 4))
plot_df = comparison[comparison['metric'].isin(['original', 'resampling_only', 'source_mamba', 'calibrated_mamba'])]
ax.bar(plot_df['metric'], plot_df['score'])
ax.axhline(1.0, color='black', linestyle='--', linewidth=1, label='dummy baseline')
ax.set_ylabel('Hyperview score lower is better')
ax.set_title(f'Sensor calibration downstream impact ({EVAL_PRESET})')
ax.tick_params(axis='x', rotation=20)
ax.legend()
plt.tight_layout()
out = DRIVE_RESULTS / f'sensor_calibration_scores_{EVAL_PRESET}.png'
fig.savefig(out, dpi=160)
print('Saved plot:', out)


## 10. Per-band bias i prediction shift

In [ ]:
val_ids = list(metrics_payload['protocol']['val_sample_ids']) if 'metrics_payload' in globals() else []
if not val_ids and predictions_path.exists():
    val_ids = sorted(predictions_df['sample_id'].astype(str).unique())
per_band_frames = []
for variant in [SOURCE_VARIANT, CALIBRATED_VARIANT, RESAMPLE_VARIANT]:
    if variant not in recon_roots:
        continue
    per_band, sample_errors = compute_per_band_diagnostics(
        original_root=HV2_ROOT,
        recon_root=recon_roots[variant],
        sample_ids=val_ids,
        original_normalization=NORMALIZATION,
        max_samples=PRESET[EVAL_PRESET]['per_band_samples'],
    )
    per_band['variant'] = variant
    per_band_frames.append(per_band)
per_band_all = pd.concat(per_band_frames, ignore_index=True) if per_band_frames else pd.DataFrame()
per_band_csv = DRIVE_RESULTS / f'per_band_calibration_diagnostics_{EVAL_PRESET}.csv'
per_band_all.to_csv(per_band_csv, index=False)
print('Saved per-band diagnostics:', per_band_csv)

summary = per_band_all.groupby('variant').agg(
    mean_mae=('mae', 'mean'),
    max_mae=('mae', 'max'),
    mean_abs_bias=('bias', lambda s: float(np.nanmean(np.abs(s)))),
    max_abs_bias=('bias', lambda s: float(np.nanmax(np.abs(s)))),
).reset_index()
display(summary.sort_values('mean_mae'))

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for variant, group in per_band_all.groupby('variant'):
    axes[0].plot(group['band'], group['mae'], label=variant)
    axes[1].plot(group['band'], group['bias'], label=variant)
axes[0].set_ylabel('MAE')
axes[1].set_ylabel('Bias')
axes[1].set_xlabel('Band')
axes[0].set_title('Per-band reconstruction error')
axes[1].axhline(0, color='black', linewidth=1)
for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
plt.tight_layout()
out = DRIVE_RESULTS / f'per_band_calibration_curves_{EVAL_PRESET}.png'
fig.savefig(out, dpi=160)
print('Saved plot:', out)

shift = prediction_shift_decomposition(predictions_df)
shift_csv = DRIVE_RESULTS / f'prediction_shift_decomposition_{EVAL_PRESET}.csv'
shift.to_csv(shift_csv, index=False)
print('Saved shift decomposition:', shift_csv)
if not shift.empty:
    display(shift[shift['variant'].isin([SOURCE_VARIANT, CALIBRATED_VARIANT, RESAMPLE_VARIANT])].sort_values(['target', 'extra_mse']).head(60))


## 11. Interpretacja koncowa

In [ ]:
print('Interpretacja:')
print('- Technika jest potwierdzona, jesli calibrated_mamba ma nizszy original->recon score niz source_mamba.')
print('- Jesli mean_abs_bias i prediction shift spadaja, problem jest glownie systematycznym shiftem widmowym.')
print('- Jesli calibrated_mamba nadal jest wyraznie gorsza niz resampling_only, sama kalibracja nie wystarcza; ale potwierdza kierunek sensor-aware/wavelength-aware adaptation.')
print('- Do pracy magisterskiej raportujemy quick jako triage, a full jako tabela finalna, jesli zdazymy uruchomic EVAL_PRESET="full".')
